Part 1

In [1]:
import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

from torch.utils.data import DataLoader
import numpy as np
import sys, time, os

/Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1016 14:56:53.784000 92090 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
###
### Part 1. Tokenization. 
###
def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]

lowercase_tokenizer("This is a test.")


['this', 'is', 'a', 'test', '.']

In [3]:

def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length. 
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """

    # TODO: build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    
    specials = [pad_token, unk_token, bos_token, eos_token]  # four special symbols
    if len(set(specials)) != 4: # Ensure all special symbols are unique.
        raise ValueError('The four special symbols must be different.')
    if max_voc_size is not None and max_voc_size < len(specials): # Ensure the vocabulary can at least hold the special symbols.
        raise ValueError('max_voc_size must be at least 4.')

    counter = Counter()
    with open(train_file, encoding='utf-8') as f:
        for line in f:  # one paragraph per nonempty line
            if line.strip():
                counter.update(tokenize_fun(line)) # count word frequencies

    for word in specials:
        counter.pop(word, None) # prevent duplicate special tokens

    n_keep = None if max_voc_size is None else max_voc_size - len(specials)
    most_common = [word for word, _ in counter.most_common(n_keep)]
    itos = specials + most_common                            # integer ID -> word
    stoi = {word: i for i, word in enumerate(itos)}          # word -> integer ID

    return A1Tokenizer(stoi, itos, tokenize_fun, model_max_length,
                       pad_token, unk_token, bos_token, eos_token)


class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""
    # TODO: store all values you need in order to implement __call__ below.
    def __init__(self, stoi, itos, tokenize_fun, model_max_length,
                 pad_token, unk_token, bos_token, eos_token):
        self.stoi = stoi
        self.itos = itos
        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length
        self.pad_token_id = stoi[pad_token]
        self.unk_token_id = stoi[unk_token]
        self.bos_token_id = stoi[bos_token]
        self.eos_token_id = stoi[eos_token]

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.

        Args:
            texts:           The texts to tokenize.
            truncation:      Whether the texts should be truncated to model_max_length.
            padding:         Whether the tokenized texts should be padded on the right side.
            return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

        Returns:
            A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """

        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # TODO: Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.

        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')

        # Handle the nested list format shown in the example.

        if texts and isinstance(texts[0], (list, tuple)):
            texts = [text for batch in texts for text in batch]


        # Split texts into words and convert them into integer IDs.
        encoded = []

        for text in texts:
            ids = [self.bos_token_id]

            for word in self.tokenize_fun(text):
                ids.append(self.stoi.get(word, self.unk_token_id))

            ids.append(self.eos_token_id)
            encoded.append(ids)

        # Truncate sequences if requested.
        if truncation and self.model_max_length is not None:

            if self.model_max_length < 2:
                raise ValueError('model_max_length must be at least 2.')

            for ids in encoded:
                if len(ids) > self.model_max_length:
                    del ids[self.model_max_length:]
                    ids[-1] = self.eos_token_id

        # Pad shorter sequences on the right if requested.
        if padding and encoded:

            max_len = max(len(ids) for ids in encoded)

            encoded = [
                ids + [self.pad_token_id] * (max_len - len(ids))
                for ids in encoded
            ]

        # Create the attention mask: 1 for real tokens, 0 for padding.
        attention_mask = [
            [1 if token_id != self.pad_token_id else 0 for token_id in ids]
            for ids in encoded
        ]

        # Convert lists into PyTorch tensors.
        if return_tensors == 'pt':
            encoded = torch.tensor(encoded, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        # Return integer-encoded texts and the attention mask.
        return BatchEncoding({
            'input_ids': encoded,
            'attention_mask': attention_mask
        })
        
    def __len__(self):
        """Return the size of the vocabulary."""
        return len(self.itos)

    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)
   

In [4]:

# =====================================================
# Part 1 — Task 1.3: Tokenizer sanity check
# =====================================================

# 1. Create the tokenizer from the training file.
# Change TRAIN_FILE to the correct path on your machine.
TRAIN_FILE = "a1_1/train.txt"

tokenizer = build_tokenizer(
    TRAIN_FILE,
    max_voc_size=10000,
    model_max_length=256
)

# 2. example.
test_texts = [['This is a test.', 'Another test.']]

result = tokenizer(
    test_texts,
    return_tensors='pt',
    padding=True,
    truncation=True
)

# 3. Inspect the results.
input_ids = result['input_ids']
attention_mask = result['attention_mask']

print("Input IDs:")
print(input_ids)

print("\nAttention mask:")
print(attention_mask)

print("\nInput shape:", input_ids.shape)
print("Mask shape:", attention_mask.shape)


# 4. Verify that both outputs are 2D tensors.
assert isinstance(input_ids, torch.Tensor)
assert isinstance(attention_mask, torch.Tensor)
assert input_ids.ndim == 2
assert input_ids.shape == attention_mask.shape
assert input_ids.shape[0] == 2

print("\nTensor shape check: PASSED")


# 5. Verify the word-to-ID encoding.
for i, text in enumerate(test_texts[0]):

    words = tokenizer.tokenize_fun(text)

    expected_ids = (
        [tokenizer.bos_token_id]
        + [tokenizer.stoi.get(w, tokenizer.unk_token_id) for w in words]
        + [tokenizer.eos_token_id]
    )

    actual_ids = input_ids[i].tolist()

    # Check BOS, words and EOS.
    assert actual_ids[:len(expected_ids)] == expected_ids

    # Check that any padding is added on the right.
    assert actual_ids[len(expected_ids):] == (
        [tokenizer.pad_token_id]
        * (len(actual_ids) - len(expected_ids))
    )

    print(f"\nText {i+1}: {text}")
    print("Words:", words)
    print("Expected IDs:", expected_ids)
    print("Actual IDs:", actual_ids)

print("\nEncoding and right-padding checks: PASSED")


# 6. Verify the attention mask.
expected_mask = (input_ids != tokenizer.pad_token_id).long()

assert torch.equal(attention_mask, expected_mask)

print("Attention mask check: PASSED")


# 7. Verify truncation with a longer text.
long_text = "This is a test. " * 100

long_result = tokenizer(
    [long_text],
    truncation=True
)

long_ids = long_result['input_ids'][0]

assert len(long_ids) <= tokenizer.model_max_length
assert long_ids[0] == tokenizer.bos_token_id
assert long_ids[-1] == tokenizer.eos_token_id

print("Truncation check: PASSED")


# 8. Save the tokenizer.
tokenizer.save("tokenizer.pkl")

# 9. Load the saved tokenizer.
loaded_tokenizer = A1Tokenizer.from_file("tokenizer.pkl")

# 10. Verify that the loaded tokenizer works identically.
loaded_result = loaded_tokenizer(
    test_texts,
    return_tensors='pt',
    padding=True,
    truncation=True
)

assert len(loaded_tokenizer) == len(tokenizer)
assert loaded_tokenizer.stoi == tokenizer.stoi
assert torch.equal(loaded_result['input_ids'], input_ids)
assert torch.equal(loaded_result['attention_mask'], attention_mask)

print("Save and reload check: PASSED")

print("\nAll Task 1.3 sanity checks passed!")




Input IDs:
tensor([[  2,  35,  14,  11, 975,   6,   3],
        [  2, 155, 975,   6,   3,   0,   0]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])

Input shape: torch.Size([2, 7])
Mask shape: torch.Size([2, 7])

Tensor shape check: PASSED

Text 1: This is a test.
Words: ['this', 'is', 'a', 'test', '.']
Expected IDs: [2, 35, 14, 11, 975, 6, 3]
Actual IDs: [2, 35, 14, 11, 975, 6, 3]

Text 2: Another test.
Words: ['another', 'test', '.']
Expected IDs: [2, 155, 975, 6, 3]
Actual IDs: [2, 155, 975, 6, 3, 0, 0]

Encoding and right-padding checks: PASSED
Attention mask check: PASSED
Truncation check: PASSED
Save and reload check: PASSED

All Task 1.3 sanity checks passed!


In [5]:

MAX_VOC_SIZE = 10000  # keep consistent with your current tokenizer build
tokenizer = build_tokenizer(
    TRAIN_FILE,
    max_voc_size=MAX_VOC_SIZE,
    model_max_length=256
)

# 1) vocab size constraint
print("vocab size:", len(tokenizer), "<= MAX_VOC_SIZE?", len(tokenizer) <= MAX_VOC_SIZE)

# 2) special tokens must exist
for special in ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]:
    print(f"{special!r} in vocab?", special in tokenizer.stoi)

# 3) frequent vs rare words
for w in ["the", "and", "sweden", "Sweden", "denmark", "Denmark", "italy", "Italy"]:
    print(f"{w!r} in vocab?", w in tokenizer.stoi)

for w in ["cuboidal", "epiglottis"]:
    print(f"{w!r} in vocab?", w in tokenizer.stoi)

# 4) round-trip check (word -> id -> word)
t = "the"
if t in tokenizer.stoi:
    tid = tokenizer.stoi[t]
    print(f"round-trip {t!r} ->", tid, "->", tokenizer.itos[tid])
else:
    print(f"{t!r} not in vocab; cannot round-trip.")


vocab size: 10000 <= MAX_VOC_SIZE? True
'<PAD>' in vocab? True
'<UNK>' in vocab? True
'<BOS>' in vocab? True
'<EOS>' in vocab? True
'the' in vocab? True
'and' in vocab? True
'sweden' in vocab? True
'Sweden' in vocab? False
'denmark' in vocab? True
'Denmark' in vocab? False
'italy' in vocab? True
'Italy' in vocab? False
'cuboidal' in vocab? False
'epiglottis' in vocab? False
round-trip 'the' -> 4 -> the


In [6]:
# =====================================================
# Part 2 — Task 2.1: Dataset sanity check
# =====================================================

# Check the number of paragraphs after removing empty lines.

from datasets import load_dataset

TRAIN_FILE = "a1_1/train.txt"
VAL_FILE = "a1_1/val.txt"

dataset = load_dataset(
    'text',
    data_files={'train': TRAIN_FILE, 'val': VAL_FILE}
)

# Remove empty lines.
dataset = dataset.filter(lambda x: x['text'].strip() != '')

#Sanity check
print("Training instances:", len(dataset['train']))
print("Validation instances:", len(dataset['val']))

print("\nFirst training example:")
print(dataset['train'][0])

print("\nFirst validation example:")
print(dataset['val'][0])

Training instances: 147059
Validation instances: 17874

First training example:
{'text': 'Anatomy'}

First validation example:
{'text': 'Andre Agassi'}


In [7]:
# Task 2.2: Create a DataLoader and inspect the first batch
from torch.utils.data import DataLoader

dl = DataLoader(
    dataset['train'],
    batch_size=4,
    shuffle=False
)

for batch in dl:
    print(batch)
    break

{'text': ['Anatomy', 'Anatomy (Greek anatomē, “dissection”) is the branch of biology concerned with the study of the structure of organisms and their parts.  Anatomy is a branch of natural science dealing with the structural organization of living things.  It is an old science, having its beginnings in prehistoric times.  Anatomy is inherently tied to embryology, comparative anatomy, evolutionary biology, and phylogeny, as these are the processes by which anatomy is generated over immediate (embryology) and long (evolution) timescales.  Human anatomy is one of the basic essential sciences of medicine.', "The discipline of anatomy is divided into macroscopic and microscopic anatomy.  Macroscopic anatomy, or gross anatomy, is the examination of an animal's body parts using unaided eyesight.  Gross anatomy also includes the branch of superficial anatomy.  Microscopic anatomy involves the use of optical instruments in the study of the tissues of various structures, known as histology, and 

# Assignment 2 — Transformer language models

In [8]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from transformers import PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutput

In [9]:
class A2ModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None, num_attention_heads=None,
                 num_hidden_layers=None, rope_theta=None, hidden_act='silu',
                 max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers


class A2MLP(nn.Module):
    """The MLP layer of the Transformer. Uses the SwiGLU architecture."""
    def __init__(self, config):
        super().__init__()
        assert config.hidden_act == 'silu'
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, hidden_states):
        gate = self.act_fn(self.gate_proj(hidden_states))
        up = self.up_proj(hidden_states)
        return self.down_proj(gate * up)


# I use nn.RMSNorm instead.


class A2Attention(nn.Module):
    """The multi-head attention layer of the Transformer. Uses scaled dot-product attention with causal masking."""
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        assert config.hidden_size % self.num_heads == 0
        self.head_dim = config.hidden_size // self.num_heads
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.o_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.q_norm = nn.RMSNorm(self.head_dim, eps=config.rms_norm_eps, elementwise_affine=True)
        self.k_norm = nn.RMSNorm(self.head_dim, eps=config.rms_norm_eps, elementwise_affine=True)

    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape
        q = self.q_proj(hidden_states).view(b, m, self.num_heads, self.head_dim)
        k = self.k_proj(hidden_states).view(b, m, self.num_heads, self.head_dim)
        v = self.v_proj(hidden_states).view(b, m, self.num_heads, self.head_dim)
        q = self.q_norm(q).transpose(1, 2)
        k = self.k_norm(k).transpose(1, 2)
        v = v.transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, rope_rotations)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).reshape(b, m, d)
        return self.o_proj(attn_out)


class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        self.self_attn = A2Attention(config)
        self.mlp = A2MLP(config)
        self.post_attention_layernorm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)
        self.post_feedforward_layernorm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)

    def forward(self, hidden_states, rope_rotations):
        attn_output = self.self_attn(hidden_states, rope_rotations)
        hidden_states = hidden_states + self.post_attention_layernorm(attn_output)
        mlp_output = self.mlp(hidden_states)
        hidden_states = hidden_states + self.post_feedforward_layernorm(mlp_output)
        return hidden_states


class A2Transformer(PreTrainedModel):
    """A language model based on the Transformer architecture."""
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)
        self.rotary_emb = A2RotaryEmbedding(config)
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([A2DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.loss_func = nn.CrossEntropyLoss(ignore_index=-100)
        self.post_init()

    def forward(self, input_ids, labels=None):
        rope_rotations = self.rotary_emb(input_ids)
        hidden_states = self.embedding(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, rope_rotations)
        hidden_states = self.norm(hidden_states)
        logits = self.unembedding(hidden_states)
        loss = None
        if labels is not None:
            loss = self.loss_func(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1)
            )
        return CausalLMOutput(logits=logits, loss=loss)


#### RoPE implementation (copied and simplified from HuggingFace). ####

def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Applies precomputed RoPE rotations to the query and key representations."""
    assert(q.shape == k.shape)
    assert(len(q.shape) == 4)
    cos, sin = rope_rotations
    assert(q.shape[2] == cos.shape[1])
    assert(q.shape[3] == cos.shape[2])
    q_type, k_type = q.dtype, k.dtype
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed.to(q_type), k_embed.to(k_type)

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

class A2RotaryEmbedding(nn.Module):
    """RoPE position representation for use in Transformer attention."""
    def __init__(self, config, device=None):
        super().__init__()
        rope_theta = config.rope_theta
        head_dim = config.hidden_size // config.num_attention_heads
        partial_rotary_factor = 1.0
        dim = int(head_dim * partial_rotary_factor)
        self.inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim))

    @torch.no_grad()
    def forward(self, x):
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()
            sin = emb.sin()
            return cos, sin

## Step 1: Checks for Tasks 1.1–1.5


In [ ]:
config = A2ModelConfig(
    vocab_size=len(tokenizer),
    hidden_size=128,
    intermediate_size=256,
    num_attention_heads=4,
    num_hidden_layers=2,
    rope_theta=10000.0,
    hidden_act='silu',
    max_position_embeddings=tokenizer.model_max_length,
    rms_norm_eps=1e-5
)
model = A2Transformer(config)



x = torch.randn(2, 6, config.hidden_size)
test_ids = torch.randint(0, len(tokenizer), (2, 6))
rope = A2RotaryEmbedding(config)(test_ids)

mlp_out = A2MLP(config)(x)
rms_out = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps)(x)
attn_out = A2Attention(config)(x, rope)
dec_out = A2DecoderLayer(config)(x, rope)

print("Condition: x shape must be (2, 6, hidden_size).")
print("Actual x shape:", x.shape)

print("Condition: test_ids shape must be (2, 6).")
print("Actual test_ids shape:", test_ids.shape)

print("Condition: rope[0]/rope[1] shapes must be (1, 6, head_dim).")
print("Actual rope cos shape:", rope[0].shape)
print("Actual rope sin shape:", rope[1].shape)

print("Condition: MLP output shape must equal x shape.")
print("Expected:", x.shape, "| Actual:", mlp_out.shape, "| Pass:", mlp_out.shape == x.shape)

print("Condition: RMSNorm output shape must equal x shape.")
print("Expected:", x.shape, "| Actual:", rms_out.shape, "| Pass:", rms_out.shape == x.shape)

print("Condition: Attention output shape must equal x shape.")
print("Expected:", x.shape, "| Actual:", attn_out.shape, "| Pass:", attn_out.shape == x.shape)

print("Condition: Decoder output shape must equal x shape.")
print("Expected:", x.shape, "| Actual:", dec_out.shape, "| Pass:", dec_out.shape == x.shape)


output = model(test_ids, labels=test_ids)

expected_logits_shape = (2, 6, len(tokenizer))
print("Condition: logits shape must be (2, 6, vocab_size).")
print("Expected:", expected_logits_shape, "| Actual:", output.logits.shape, "| Pass:", output.logits.shape == expected_logits_shape)

print("Condition: loss must be finite.")
print("Actual loss shape:", output.loss.shape)
print("Actual loss value:", output.loss.item())
print("Pass:", torch.isfinite(output.loss).item())

# Keep hard checks.
assert mlp_out.shape == x.shape
assert rms_out.shape == x.shape
assert attn_out.shape == x.shape
assert dec_out.shape == x.shape
assert output.logits.shape == expected_logits_shape
assert torch.isfinite(output.loss)

print("All shape and loss checks: PASSED")

Condition: x shape must be (2, 6, hidden_size).
Actual x shape: torch.Size([2, 6, 128])
Condition: test_ids shape must be (2, 6).
Actual test_ids shape: torch.Size([2, 6])
Condition: rope[0]/rope[1] shapes must be (1, 6, head_dim).
Actual rope cos shape: torch.Size([1, 6, 32])
Actual rope sin shape: torch.Size([1, 6, 32])
Condition: MLP output shape must equal x shape.
Expected: torch.Size([2, 6, 128]) | Actual: torch.Size([2, 6, 128]) | Pass: True
Condition: RMSNorm output shape must equal x shape.
Expected: torch.Size([2, 6, 128]) | Actual: torch.Size([2, 6, 128]) | Pass: True
Condition: Attention output shape must equal x shape.
Expected: torch.Size([2, 6, 128]) | Actual: torch.Size([2, 6, 128]) | Pass: True
Condition: Decoder output shape must equal x shape.
Expected: torch.Size([2, 6, 128]) | Actual: torch.Size([2, 6, 128]) | Pass: True
Condition: logits shape must be (2, 6, vocab_size).
Expected: (2, 6, 10000) | Actual: torch.Size([2, 6, 10000]) | Pass: True
Condition: loss must 

In [11]:
#remove it


# Dummy batch
input_ids = torch.randint(0, config.vocab_size, (2, 6))
labels = input_ids.clone()

B, T = input_ids.shape
D = config.hidden_size
H = config.num_attention_heads
HD = D // H
V = config.vocab_size

def show(name, actual, expected):
    print(f"Condition: {name} shape must be {expected}.")
    print(f"Actual {name} shape: {tuple(actual.shape)} | Pass: {tuple(actual.shape) == tuple(expected)}")

# 1) Input
show("input_ids", input_ids, (B, T))

# 2) RoPE cache
rope = model.rotary_emb(input_ids)
cos, sin = rope
show("rope.cos", cos, (1, T, HD))
show("rope.sin", sin, (1, T, HD))

# 3) Token embedding
hidden_states = model.embedding(input_ids)
show("embedding output", hidden_states, (B, T, D))

# 4) Decoder layers (OLMo order)
for i, layer in enumerate(model.layers):
    print(f"\n===== Decoder layer {i} =====")
    x_in = hidden_states
    show(f"layer{i}.input", x_in, (B, T, D))

    # Attention sublayer internals
    q_lin = layer.self_attn.q_proj(x_in)
    k_lin = layer.self_attn.k_proj(x_in)
    v_lin = layer.self_attn.v_proj(x_in)
    show(f"layer{i}.q_proj", q_lin, (B, T, D))
    show(f"layer{i}.k_proj", k_lin, (B, T, D))
    show(f"layer{i}.v_proj", v_lin, (B, T, D))

    q = q_lin.view(B, T, H, HD)
    k = k_lin.view(B, T, H, HD)
    v = v_lin.view(B, T, H, HD)
    show(f"layer{i}.q_view", q, (B, T, H, HD))
    show(f"layer{i}.k_view", k, (B, T, H, HD))
    show(f"layer{i}.v_view", v, (B, T, H, HD))

    q = layer.self_attn.q_norm(q).transpose(1, 2)  # (B,H,T,HD)
    k = layer.self_attn.k_norm(k).transpose(1, 2)  # (B,H,T,HD)
    v = v.transpose(1, 2)                           # (B,H,T,HD)
    show(f"layer{i}.q_norm+transpose", q, (B, H, T, HD))
    show(f"layer{i}.k_norm+transpose", k, (B, H, T, HD))
    show(f"layer{i}.v_transpose", v, (B, H, T, HD))

    q_rot, k_rot = apply_rotary_pos_emb(q, k, rope)
    show(f"layer{i}.q_rot", q_rot, (B, H, T, HD))
    show(f"layer{i}.k_rot", k_rot, (B, H, T, HD))

    attn_heads = F.scaled_dot_product_attention(q_rot, k_rot, v, is_causal=True)  # (B,H,T,HD)
    show(f"layer{i}.attn_heads", attn_heads, (B, H, T, HD))

    attn_merge = attn_heads.transpose(1, 2).reshape(B, T, D)
    show(f"layer{i}.attn_merge", attn_merge, (B, T, D))

    attn_proj = layer.self_attn.o_proj(attn_merge)
    show(f"layer{i}.attn_o_proj", attn_proj, (B, T, D))

    # Residual + norm after attention
    x_after_attn = x_in + layer.post_attention_layernorm(attn_proj)
    show(f"layer{i}.residual_after_attn", x_after_attn, (B, T, D))

    # MLP sublayer internals
    gate_lin = layer.mlp.gate_proj(x_after_attn)
    up_lin = layer.mlp.up_proj(x_after_attn)
    gate_act = layer.mlp.act_fn(gate_lin)
    swiglu_mul = gate_act * up_lin
    mlp_down = layer.mlp.down_proj(swiglu_mul)

    show(f"layer{i}.mlp_gate_proj", gate_lin, (B, T, config.intermediate_size))
    show(f"layer{i}.mlp_up_proj", up_lin, (B, T, config.intermediate_size))
    show(f"layer{i}.mlp_gate_act", gate_act, (B, T, config.intermediate_size))
    show(f"layer{i}.mlp_swiglu_mul", swiglu_mul, (B, T, config.intermediate_size))
    show(f"layer{i}.mlp_down_proj", mlp_down, (B, T, D))

    # Residual + norm after MLP
    hidden_states = x_after_attn + layer.post_feedforward_layernorm(mlp_down)
    show(f"layer{i}.output", hidden_states, (B, T, D))

# 5) Final norm + LM head
hidden_states = model.norm(hidden_states)
show("final_norm output", hidden_states, (B, T, D))

logits = model.unembedding(hidden_states)
show("logits", logits, (B, T, V))

# 6) Loss (teacher-forcing shift)
loss = model.loss_func(
    logits[:, :-1, :].reshape(-1, logits.size(-1)),
    labels[:, 1:].reshape(-1)
)
print("Condition: loss must be a finite scalar.")
print("Actual loss shape:", tuple(loss.shape), "| Pass shape==():", tuple(loss.shape) == ())
print("Actual loss value:", loss.item(), "| Pass finite:", torch.isfinite(loss).item())

# Optional hard checks
assert logits.shape == (B, T, V)
assert torch.isfinite(loss)

print("\nAll ordered architecture checks: PASSED")


Condition: input_ids shape must be (2, 6).
Actual input_ids shape: (2, 6) | Pass: True
Condition: rope.cos shape must be (1, 6, 32).
Actual rope.cos shape: (1, 6, 32) | Pass: True
Condition: rope.sin shape must be (1, 6, 32).
Actual rope.sin shape: (1, 6, 32) | Pass: True
Condition: embedding output shape must be (2, 6, 128).
Actual embedding output shape: (2, 6, 128) | Pass: True

===== Decoder layer 0 =====
Condition: layer0.input shape must be (2, 6, 128).
Actual layer0.input shape: (2, 6, 128) | Pass: True
Condition: layer0.q_proj shape must be (2, 6, 128).
Actual layer0.q_proj shape: (2, 6, 128) | Pass: True
Condition: layer0.k_proj shape must be (2, 6, 128).
Actual layer0.k_proj shape: (2, 6, 128) | Pass: True
Condition: layer0.v_proj shape must be (2, 6, 128).
Actual layer0.v_proj shape: (2, 6, 128) | Pass: True
Condition: layer0.q_view shape must be (2, 6, 4, 32).
Actual layer0.q_view shape: (2, 6, 4, 32) | Pass: True
Condition: layer0.k_view shape must be (2, 6, 4, 32).
Actual

## Step 2 — Task 2.1: Training


In [12]:


## Hint: the following TrainingArguments hyperparameters may be relevant for your implementation:
#
# - optim:            What optimizer to use. You can assume that this is set to 'adamw_torch',
#                     meaning that we use the PyTorch AdamW optimizer.
# - eval_strategy:    You can assume that this is set to 'epoch', meaning that the model should
#                     be evaluated on the validation set after each epoch
# - use_cpu:          Force the trainer to use the CPU; otherwise, CUDA or MPS should be used.
#                     (In your code, you can just use the provided method select_device.)
# - learning_rate:    The optimizer's learning rate.
# - num_train_epochs: The number of epochs to use in the training loop.
# - per_device_train_batch_size: 
#                     The batch size to use while training.
# - per_device_eval_batch_size:
#                     The batch size to use while evaluating.
# - output_dir:       The directory where the trained model will be saved.

class A1Trainer:
    """A minimal implementation similar to a Trainer from the HuggingFace library."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        """Set up the trainer.
           
           Args:
             model:          The model to train.
             args:           The training parameters stored in a TrainingArguments object.
             train_dataset:  The dataset containing the training documents.
             eval_dataset:   The dataset containing the validation documents.
             eval_dataset:   The dataset containing the validation documents.
             tokenizer:      The tokenizer.
        """
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer

        assert(args.optim == 'adamw_torch')
        assert(args.eval_strategy == 'epoch')

    def select_device(self):
        if self.args.use_cpu:
            return torch.device('cpu')
        no_cuda = getattr(self.args, "no_cuda", False)
        if (not no_cuda) and torch.cuda.is_available():
            return torch.device('cuda')
        if torch.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')

        
            
    def train(self):
        """Train the model."""
        args = self.args

        device = self.select_device()
        print('Device:', device)
        self.model.to(device)
        
        #loss_func = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id) # The model itself already computes loss with ignore_index=-100.

        # TODO: Relevant arguments: at least args.learning_rate, but you can optionally also consider
        # other Adam-related hyperparameters here.
        # optimizer = torch.optim.AdamW(...)


        # TODO: Relevant arguments: args.per_device_train_batch_size, args.per_device_eval_batch_size
        # train_loader = DataLoader(...)
        # val_loader = DataLoader(...)
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=args.learning_rate)
        train_loader = DataLoader(self.train_dataset, batch_size=args.per_device_train_batch_size,
                                  shuffle=True)
        val_loader = DataLoader(self.eval_dataset, batch_size=args.per_device_eval_batch_size,
                                shuffle=False)




        # TODO: Your work here is to implement the training loop.
        #       
        # for each training epoch (use args.num_train_epochs here):
        #   for each batch B in the training set:
        #
        #       PREPROCESSING AND FORWARD PASS:
        #       input_ids = apply your tokenizer to B
        #       labels = input_ids with padding replaced by -100
	    #       put input_ids and labels onto the GPU (or whatever device you use)
        #       apply the model to input_ids and labels
        #       get the loss from the model output
        #
        #       BACKWARD PASS AND MODEL UPDATE:
        #       optimizer.zero_grad()
        #       loss.backward()
        #       optimizer.step()


        num_epochs = int(args.num_train_epochs)
        if num_epochs < 1 or num_epochs != args.num_train_epochs:
            raise ValueError('This minimal trainer requires a positive whole number of epochs.')

        for epoch in range(num_epochs):
            self.model.train()
            for batch in train_loader:
                # Tokenize and mark padded labels with -100 (not used in the loss).
                enc = self.tokenizer(batch['text'], truncation=True, padding=True, return_tensors='pt')
                input_ids = enc['input_ids']
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
                input_ids, labels = input_ids.to(device), labels.to(device)

                output = self.model(input_ids=input_ids, labels=labels)  # forward + loss
                optimizer.zero_grad()                                   # clear old gradients
                output.loss.backward()                                  # compute gradients
                optimizer.step()                                        # update parameters

            self.model.eval()
            total_loss, total_tokens = 0.0, 0
            with torch.no_grad():
                for batch in val_loader:
                    enc = self.tokenizer(batch['text'], truncation=True, padding=True, return_tensors='pt')
                    input_ids = enc['input_ids']
                    labels = input_ids.clone()
                    labels[labels == self.tokenizer.pad_token_id] = -100
                    input_ids, labels = input_ids.to(device), labels.to(device)

                    output = self.model(input_ids=input_ids, labels=labels)
                    n = (labels[:, 1:] != -100).sum().item()            # valid next tokens
                    total_loss += output.loss.item() * n                # weight by token count
                    total_tokens += n

            if total_tokens == 0:
                raise ValueError('No valid target tokens in the validation set.')
            eval_loss = total_loss / total_tokens
            eval_ppl = math.exp(eval_loss)
            print(f'epoch {epoch + 1}: eval_loss={eval_loss:.4f}, eval_ppl={eval_ppl:.4f}')


        print(f'Saving to {args.output_dir}.')
        self.model.save_pretrained(args.output_dir)


In [14]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir='a2_transformer_model',
    learning_rate=1e-3,
    num_train_epochs=3.0,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    optim='adamw_torch',
    eval_strategy='epoch',
    use_cpu=False,
)

trainer = A1Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    tokenizer=tokenizer,
)
trainer.train()

Device: mps
epoch 1: eval_loss=4.2624, eval_ppl=70.9817
epoch 2: eval_loss=4.1333, eval_ppl=62.3827
epoch 3: eval_loss=4.0839, eval_ppl=59.3787
Saving to a2_transformer_model.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 42.71it/s]


## Step 2 — Task 2.1: Validation perplexity


In [15]:
# Part 5.2 — Perplexity on validation set
import math
from torch.utils.data import DataLoader

def compute_perplexity(model, tokenizer, eval_dataset, batch_size=32):
    model.eval()
    device = next(model.parameters()).device
    val_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in val_loader:
            enc = tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt")
            input_ids = enc["input_ids"].to(device)

            labels = input_ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100

            out = model(input_ids=input_ids, labels=labels)
            n_valid = (labels[:, 1:] != -100).sum().item()  # valid next-token targets

            if n_valid > 0:
                total_loss += out.loss.item() * n_valid
                total_tokens += n_valid

    if total_tokens == 0:
        raise ValueError("No valid target tokens in eval set.")

    mean_ce = total_loss / total_tokens
    ppl = math.exp(mean_ce)
    return mean_ce, ppl


In [ ]:

mean_ce, ppl = compute_perplexity(model, tokenizer, dataset["val"], batch_size=32)
print(f"Validation cross-entropy: {mean_ce:.4f}")
print(f"Validation perplexity: {ppl:.2f}")

Validation cross-entropy: 4.0738
Validation perplexity: 58.78


## Step 3 — Task 3.1: Predict the next word


In [ ]:

import torch

def predict_next_word(model, tokenizer, text, topk=5, truncation=True):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer([text], truncation=truncation, padding=False, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)

    with torch.no_grad():
        out = model(input_ids=input_ids)
        logits = out.logits  # (1, N, V)

    # The tokenizer adds EOS, so use the second-to-last position.
    scores = logits[0, -2, :]  # (V,)

    # Find the highest-scoring candidates.
    k = min(topk, scores.shape[0])
    topv, topi = torch.topk(scores, k=k)

    # Map predicted IDs back to words.
    preds = [
        (tokenizer.itos[i.item()], score.item())
        for i, score in zip(topi, topv)
    ]

    # Find the single best next word.
    best_id = torch.argmax(scores).item()
    best_word = tokenizer.itos[best_id]

    return best_word, preds


# some tests
best, top_preds = predict_next_word(
    model, tokenizer, "She lives in San", topk=10
)

print("Best prediction:", best)
print("\nTop 10 predictions:")

for word, score in top_preds:
    print(f"{word}: {score:.4f}")


Best prediction: diego

Top 10 predictions:
diego: 13.2036
<UNK>: 11.6914
francisco: 11.6414
suu: 10.7202
salvador: 10.4989
juan: 9.8592
antonio: 9.0622
carlos: 8.7334
pedro: 7.4948
rosa: 7.4694


In [17]:
best, top_preds = predict_next_word(
    model, tokenizer, "The president of the united", topk=10
)

print("Best prediction:", best)
print("\nTop 10 predictions:")

Best prediction: states

Top 10 predictions:


## Step 3 — Task 3.2: Random generation with temperature and top-K


In [18]:
@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0, topk=None):
    model.eval()
    device = next(model.parameters()).device
    input_ids = tokenizer([prompt], truncation=True, return_tensors='pt')['input_ids'].to(device)
    input_ids = input_ids[:, :-1]  # custom A1Tokenizer appends EOS to every input
    new_words = []

    for _ in range(max_length):
        logits = model(input_ids=input_ids).logits[:, -1, :] / temperature

        if topk is not None:
            top_vals, top_idx = torch.topk(logits, k=min(topk, logits.size(-1)))
            filtered = torch.full_like(logits, float('-inf'))
            filtered.scatter_(1, top_idx, top_vals)
            logits = filtered

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

        input_ids = torch.cat([input_ids, next_token], dim=1)
        new_words.append(tokenizer.itos[next_token.item()])

    return prompt + (' ' + ' '.join(new_words) if new_words else '')


prompt = 'In natural language processing, a Transformer'
print(generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0))
print(generate_text(model, tokenizer, prompt, max_length=50, temperature=0.7, topk=20))
print(generate_text(model, tokenizer, 'Is Stockholm the capital of Sweden? Answer yes or no. The answer is', max_length=50, temperature=1.0, topk=20))
print(generate_text(model, tokenizer, 'Write a Python program that reverses a list.', max_length=50, temperature=1.0, topk=20))

In natural language processing, a Transformer and topic by cultural production as a notorious <UNK> , the <UNK> being a clear chance of a spanish <UNK> essentially better came to many new <UNK> agents . their goals seen as a <UNK> process of persistent bilateral , successive agents , and signal labor . <UNK> <UNK> are
In natural language processing, a Transformer <UNK> with <UNK> and <UNK> , also called <UNK> , is a <UNK> language that uses <UNK> ( <UNK> ) , and <UNK> ( <UNK> ) or <UNK> ( <UNK> ) . it also uses <UNK> and <UNK> .
Is Stockholm the capital of Sweden? Answer yes or no. The answer is the list of other people . it is also <UNK> . however , some critics use the possibility for the use of the <UNK> , but this does not specify the <UNK> of <UNK> . it seems that the <UNK> of the <UNK> of the <UNK> was no better in
Write a Python program that reverses a list. in this way the data from a <UNK> of data to a <UNK> <UNK> in the <UNK> module on this program , or the <UNK> <UNK> is cal

## Step 3 — Task 3.3: Compare with a pretrained Transformer (mistral:7b)

In [21]:
# Task 3.3 — Comparing to a pretrained Transformer baseline via Ollama

# If the model is not installed locally yet, run this in the terminal:
# ollama pull mistral:7b

try:
    import ollama
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Ollama Python package is missing. Install with:\n"
        "python -m pip install ollama"
    ) from e

MODEL_NAME = "mistral:7b"

def generate_ollama_text(prompt, model_name=MODEL_NAME, max_new_tokens=40, temperature=0.8):
    response = ollama.generate(
        model=model_name,
        prompt=prompt,
        options={
            "temperature": temperature,
            "top_k": 20,
            "num_predict": max_new_tokens,
        },
    )
    return response["response"]

for prompt in [
    "In natural language processing, a Transformer",
    "Is Stockholm the capital of Sweden? Answer yes or no. The answer is",
    "Write a Python program that reverses a list.",
]:
    print("\nPROMPT:", prompt)
    print("Pretrained Ollama model:", MODEL_NAME)
    print(generate_ollama_text(prompt, model_name=MODEL_NAME, max_new_tokens=40, temperature=0.8))


PROMPT: In natural language processing, a Transformer
Pretrained Ollama model: mistral:7b
 In natural language processing (NLP), a Transformer is a type of model architecture that's particularly effective for tasks like translation, text summarization, and text generation. It was introduced in a

PROMPT: Is Stockholm the capital of Sweden? Answer yes or no. The answer is
Pretrained Ollama model: mistral:7b
 Yes. Stockholm is the capital of Sweden.

PROMPT: Write a Python program that reverses a list.
Pretrained Ollama model: mistral:7b
 Here's a simple Python program that reverses a given list:

```python
def reverse_list(input_list):
    return input_list[::-1]


## my reflection:
The pretrained model (mistral:7b) is answering correctly to the questions while the one that we trained in this homework answered wrongly to the prompts.